In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e11/sample_submission.csv
/kaggle/input/playground-series-s5e11/train.csv
/kaggle/input/playground-series-s5e11/test.csv
/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv


In [2]:
N_FOLDS = 5
SEED = 42

In [3]:
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
train_org = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
TARGET = train.columns[-1]
test[TARGET] = -1
# combine

In [4]:
FEATURES = list(train.columns[1:-1])
print(f'FEATURES_{len(FEATURES)}: {FEATURES}, TARGET: {TARGET}')

FEATURES_11: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], TARGET: loan_paid_back


In [5]:
train_org = train_org[FEATURES + [TARGET]]
train_org[TARGET] = train_org[TARGET].astype('float64')

# train_org = train_org[FEATURES + [TARGET]].reset_index(drop=True)
# train     = train[FEATURES + [TARGET]].reset_index(drop=True)
# test      = test[FEATURES].reset_index(drop=True)         

# n_org = len(train_org)
# n_tr  = len(train)
# n_te  = len(test)

combine = pd.concat([train_org, train.drop(columns='id'), test.drop(columns='id')], axis=0)

In [6]:
CATS = []
NUMS = []
for c in FEATURES:
    t='CAT'
    if train[c].dtype=='object':
        CATS.append(c)
    else:
        NUMS.append(c)
        t='NUM'

    n = train[c].nunique()
    na = train[c].isna().sum()
    print(f'[{t}] {c} has {n} unique and {na} NA')

print('CATS:', CATS)
print('NUMS:', NUMS)

[NUM] annual_income has 119728 unique and 0 NA
[NUM] debt_to_income_ratio has 526 unique and 0 NA
[NUM] credit_score has 399 unique and 0 NA
[NUM] loan_amount has 111570 unique and 0 NA
[NUM] interest_rate has 1454 unique and 0 NA
[CAT] gender has 3 unique and 0 NA
[CAT] marital_status has 4 unique and 0 NA
[CAT] education_level has 5 unique and 0 NA
[CAT] employment_status has 5 unique and 0 NA
[CAT] loan_purpose has 8 unique and 0 NA
[CAT] grade_subgrade has 30 unique and 0 NA
CATS: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMS: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [7]:
# from itertools import combinations
# from tqdm import tqdm

# for col1, col2 in tqdm(combinations(CATS, 2)):
#     new_col_name = f'{col1}_{col2}'
#     combine[new_col_name] = combine[col1].astype(str) + '_' + combine[col2].astype(str)
#     CATS.append(new_col_name)

In [8]:
CATS1 = []
SIZES = {}

for c in CATS+NUMS:
    n=c
    if c in NUMS:
        n=f'{c}2'
        CATS1.append(n)
    combine[n],_ = combine[c].factorize()
    SIZES[n] = combine[n].max()+1

    combine[c] = combine[c].astype('int32')
    combine[n] = combine[n].astype('int32')

print('NEW CATS:', CATS1)
print('CARDINALITY OF ALL CATS:', SIZES)

NEW CATS: ['annual_income2', 'debt_to_income_ratio2', 'credit_score2', 'loan_amount2', 'interest_rate2']
CARDINALITY OF ALL CATS: {'gender': 3, 'marital_status': 4, 'education_level': 5, 'employment_status': 5, 'loan_purpose': 8, 'grade_subgrade': 30, 'annual_income2': 152960, 'debt_to_income_ratio2': 572, 'credit_score2': 406, 'loan_amount2': 139138, 'interest_rate2': 1498}


In [9]:
# from itertools import combinations 

# pairs = combinations(CATS + CATS1, 2)
# new_cols = {}
# CATS2 = []

# for c1, c2 in pairs:
#     name = '_'.join(sorted((c1, c2)))
#     new_cols[name] = combine[c1] * SIZES[c2] + combine[c2]
#     CATS2.append(name)

# if new_cols:
#     new_df = pd.DataFrame(new_cols)
#     combine = pd.concat([combine, new_df], axis=1)

# print(f'Created {len(CATS2)} new CAT columns')

In [10]:
train_org_n = combine.iloc[:len(train_org)]
train_n = combine.iloc[len(train_org):len(train)+len(train_org)]
test_n = combine.iloc[len(train)+len(train_org):]

In [11]:
TE = []
for c in CATS:
    tmp = train_org_n.groupby(c)[TARGET].mean()
    tmp_sum = train_org_n.groupby(c)[TARGET].sum()
    tmp_cnt = train_org_n.groupby(c).size()
    tmp_neg = tmp_cnt - tmp_sum

    global_mean = train_org_n[TARGET].mean()
    loo = (tmp * tmp_cnt - train_org_n[TARGET]) / (tmp_cnt - 1)
    loo = loo.fillna(global_mean)
    loo.name = f'TE_loo_{c}'
    
    # 6. log-odds  (helps linear / NN models)
    
    odds = np.log((tmp + 1e-5) / (1 - tmp + 1e-5))
    odds.name = f'TE_logodds_{c}'
    
    # 7. group variance of target  (0-1 variance)
    tmp_var = train_org_n.groupby(c)[TARGET].var()
    tmp_var.name = f'TE_var_{c}'
    
    # 8. smoothed / Bayesian target encoding
    alpha = 20
    smooth = (tmp * tmp_cnt + global_mean * alpha) / (tmp_cnt + alpha)
    smooth.name = f'TE_smooth_{c}'
    
    # 9. chi-square deviation  (absolute difference from global mean)
    dev = (tmp - global_mean).abs()
    dev.name = f'TE_dev_{c}'
    
    n = f'TE_{c}'
    n_s = f'TE_sum_{c}'
    n_c = f'TE_count_{c}'
    n_neg = f'TE_neg_{c}'
    print(f'{n} , {n_c}', end='')
    tmp.name = n
    tmp_cnt.name = n_c
    tmp_sum.name = n_s
    tmp_neg.name = n_neg

    stats = (pd.concat([tmp, 
                       # tmp_cnt,
                       tmp_sum,
                       # loo,
                       # tmp_neg
                      ]
                      , axis=1,).reset_index().rename(columns={'index': c})) 
    train_org_n = train_org_n.merge(stats, on=c, how='left')
    # train_org_n = train_org_n.merge(tmp_cnt, on=c, how='left')
    
    train_n = train_n.merge(stats, on=c, how='left')
    # train_n = train_n.merge(tmp_cnt, on=c, how='left')
    
    test_n = test_n.merge(stats, on=c, how='left')
    # test_n = test_n.merge(tmp_cnt, on=c, how='left')
    
    TE.append(n)
    TE.append(n_c)

TE_gender , TE_count_genderTE_marital_status , TE_count_marital_statusTE_education_level , TE_count_education_levelTE_employment_status , TE_count_employment_statusTE_loan_purpose , TE_count_loan_purposeTE_grade_subgrade , TE_count_grade_subgrade

In [12]:
train_n.isna().sum()

annual_income               0
debt_to_income_ratio        0
credit_score                0
loan_amount                 0
interest_rate               0
gender                      0
marital_status              0
education_level             0
employment_status           0
loan_purpose                0
grade_subgrade              0
loan_paid_back              0
annual_income2              0
debt_to_income_ratio2       0
credit_score2               0
loan_amount2                0
interest_rate2              0
TE_gender                   0
TE_sum_gender               0
TE_marital_status           0
TE_sum_marital_status       0
TE_education_level          0
TE_sum_education_level      0
TE_employment_status        0
TE_sum_employment_status    0
TE_loan_purpose             0
TE_sum_loan_purpose         0
TE_grade_subgrade           0
TE_sum_grade_subgrade       0
dtype: int64

In [13]:
train_n.drop(columns=[TARGET]).columns

Index(['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount',
       'interest_rate', 'gender', 'marital_status', 'education_level',
       'employment_status', 'loan_purpose', 'grade_subgrade', 'annual_income2',
       'debt_to_income_ratio2', 'credit_score2', 'loan_amount2',
       'interest_rate2', 'TE_gender', 'TE_sum_gender', 'TE_marital_status',
       'TE_sum_marital_status', 'TE_education_level', 'TE_sum_education_level',
       'TE_employment_status', 'TE_sum_employment_status', 'TE_loan_purpose',
       'TE_sum_loan_purpose', 'TE_grade_subgrade', 'TE_sum_grade_subgrade'],
      dtype='object')

In [14]:
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import PolynomialFeatures, OrdinalEncoder, PowerTransformer, QuantileTransformer
from sklearn.metrics import r2_score
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import warnings
import category_encoders as ce
warnings.filterwarnings('ignore')
from sklearn.metrics import roc_auc_score

def cv_score(model_dict, X, y, test, n_folds=N_FOLDS, seed=SEED, train_org=None):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores_dict = {}
    cat_feats = X.select_dtypes(include=['object', 'category']).columns
    num_feats = X.select_dtypes(include=['int64', 'float64', 'bool']).columns
    cols = X.columns
    # cat_feats = X.columns.to_list()
    oof_preds_dict = {}
    test_preds_dict = {}
    for name, model in model_dict.items():
        scores = []
        r2_scores = []
        fold_test_preds = np.zeros(len(test))
        oof_preds = np.zeros(len(X))
        test_preds = np.zeros(len(test))
        for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
            # poly = PolynomialFeatures(include_bias=True, degree=2)
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            test_n = test.copy()
            if train_org is not None:
                train_orig_n = train_org.copy()
                train_orig_n = pd.concat([train_orig_n]*1, axis=0)
                print(f'ORGINAL DATA SHAPE: {len(train_orig_n)}')
                X_train = pd.concat([X_train, train_orig_n.drop(columns=TARGET)], axis=0)
                y_train = pd.concat([y_train, train_orig_n[TARGET]], axis=0)

           
            
            print(f'updated len of columns in train:{X_train.shape[1]} , test:{test_n.shape[1]}')
            clf = clone(model)

            clf.fit(X_train, y_train,
                eval_set=[(X_val, y_val)],
                # early_stopping_rounds=200,
                verbose=300
                   )
            val_preds = clf.predict_proba(X_val)[:,1]

            oof_preds[val_idx] = val_preds 
            print(f'TEST SHAPE: {test_n[num_feats].shape}')
            preds = clf.predict_proba(test_n)[:,1]
                
            fold_test_preds += (preds)
            score = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
            # r2 = r2_score(y.iloc[val_idx], oof_preds[val_idx])
            # r2_scores.append(r2)
            scores.append(score)
            print(F'ROC SCORE FOR FOLD{i}: [{score}]')
    

        test_preds = fold_test_preds / N_FOLDS
        print(f'Mean score for model {name} across all folds : {np.mean(scores)}')
        oof_preds_dict[name] = oof_preds
        test_preds_dict[name] = test_preds
        scores_dict[name] = scores

    return scores_dict, oof_preds_dict, test_preds_dict
xgb_params = {
    'n_estimators': 4000,
    'max_depth': 15,
    'learning_rate': 0.015216251287478466,
    'subsample': 0.6869288537951837,
    'colsample_bytree': 0.939877058046764,
    'reg_alpha': 4.883004566524303e-06,
    'reg_lambda': 0.0016477801911808878,
    'min_child_weight': 8,
    'gamma': 0.009579430708897484,
    'random_state': 42,
    'n_jobs': -1,
    'enable_categorical': True,
    'tree_method': 'hist'  
}
lgb_params = {  "objective"            : "binary",
                       "data_sample_strategy" : "goss",
                       "eval_metric"          : "auc",
                       'device'               : "gpu" ,
                       'learning_rate'        : 0.01,
                       'n_estimators'         : 12000 ,
                       'max_depth'            : 6,
                       'subsample'            : 0.825,
                       'colsample_bytree'     : 0.55,
                       'reg_lambda'           : 0.85,
                       'reg_alpha'            : 0.001,
                       'verbosity'            : -1,
                       'random_state'         : 42,
                      } 
       
xgb_sk_params = {
    # 'objective': 'binary:logistic',
    # 'eval_metric': 'auc',
    # 'max_depth': 5,
    # 'colsample_bytree': 0.8,
    # 'subsample': 0.8,
    # 'n_estimators': 100000,
    # 'learning_rate': 0.01,
    # # 'early_stopping_rounds': 100,
    # 'random_state': 42,
    # 'n_jobs': -1,
    # 'device': 'cuda',
    # 'enable_categorical': True,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 100000,
    'learning_rate': 0.1,
    'early_stopping_rounds': 500,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}

model_dict = {
#    'lgb':lgb.LGBMRegressor(
#     n_estimators=12_000,        # many *weak* trees
#     learning_rate=0.003,        # tiny steps
#     num_leaves=15,              # very coarse splits
#     max_depth=-1,               # let leaves decide
#     min_data_in_leaf=15_000,    # ≥ 3 % of data per leaf
#     feature_fraction=0.20,      # 1/5 of cols per tree
#     bagging_fraction=0.50,      # 1/2 of rows per tree
#     lambda_l1=2.0,              # strong L1
#     lambda_l2=2.0,              # strong L2
#     random_state=SEED,
#     verbosity=-1,
#     device='GPU'
# )
    'xgb': xgb.XGBClassifier(**xgb_sk_params),
    # 'lgb': lgb.LGBMClassifier(**lgb_params)
    # 'cb': cb.CatBoostRegressor(
    # depth                   = 15,
    # iterations              = 1_500,
    # learning_rate           = 0.039,
    # l2_leaf_reg             = 0.0001,      # LGB reg_lambda analogue
    # random_seed             = SEED,
    # # rsm                     = 1.0,         # feature_fraction analogue
    # # subsample               = 1.0,
    # bootstrap_type     ernoulli', # needed when subsample < 1
    # task_type               = 'GPU',
    # verbose                 = 100          # print every 100 trees
    # ),
    # 'xgb': xgb.XGBRegressor(
    # max_depth               = 15,
    # n_estimators            = 1_500,
    # learning_rate           = 0.039,
    # reg_alpha               = 0.0001,      # L1
    # reg_lambda              = 0.0001,      # L2
    # # subsample               = 1.0,         # LGB default 1.0 when not tuned
    # # colsample_bytree        = 1.0,         # LGB “feature_fraction” analogue
    # random_state            = SEED,
    # tree_method             = 'gpu_hist',  # GPU training
    # objective               = 'reg:squarederror',
    # # enable_categorical      = True, 
    # n_jobs                  = -1,
    # verbose                 = 0
    # # verbosity               = 0
    # ),
    # 'etr': ExtraTreesRegressor(n_estimators=400, max_depth=60, n_jobs=-1)
    # 'lgb':lgb.LGBMRegressor(
    #             max_depth           = 15,
    #             num_leaves          = 65, 
    #             n_estimators        = 1_500 , 
    #             learning_rate       = 0.039,
    #             # feature_fraction    = 0.80, 
    #             # subsample           = 0.85,
    #             reg_alpha           = 0.001,
    #             reg_lambda          = 0.001, 
    #             random_state        = SEED,
    #             verbosity           = 1,
    #             device              = 'gpu'
    #              )
                 # {"callbacks"   : [log_evaluation(0)],
                 #  "eval_metric" : "rmse",
                 # } 
    #          ],
    #     # n_esimators=8000,
    #     # learning_rate=0.1,
    #     random_state=SEED,
    #     verbosity=-1
    # ),
    # 'lr':LinearRegression(n_jobs=-1),
    # 'br':BayesianRidge()
 }

# scores_dict, oof_preds, test_preds = cv_score(model_dict, train[.drop(columns=['id', TARGET])], train[TARGET], test.drop(columns=['id', TARGET]), train_org=None)
scores_dict, oof_preds, test_preds = cv_score(model_dict, train_n.drop(columns=TARGET), train_n[TARGET], test_n.drop(columns=TARGET), train_org=None)

updated len of columns in train:28 , test:28
[0]	validation_0-auc:0.85478
[300]	validation_0-auc:0.90487
[600]	validation_0-auc:0.90620
[900]	validation_0-auc:0.90616
[1168]	validation_0-auc:0.90595
TEST SHAPE: (254569, 12)
ROC SCORE FOR FOLD1: [0.9062763847457113]
updated len of columns in train:28 , test:28
[0]	validation_0-auc:0.85418
[300]	validation_0-auc:0.90565
[600]	validation_0-auc:0.90759
[900]	validation_0-auc:0.90775
[1193]	validation_0-auc:0.90757
TEST SHAPE: (254569, 12)
ROC SCORE FOR FOLD2: [0.9078330681538059]
updated len of columns in train:28 , test:28
[0]	validation_0-auc:0.85434
[300]	validation_0-auc:0.90254
[600]	validation_0-auc:0.90440
[900]	validation_0-auc:0.90446
[1200]	validation_0-auc:0.90409
[1341]	validation_0-auc:0.90386
TEST SHAPE: (254569, 12)
ROC SCORE FOR FOLD3: [0.9045276089827454]
updated len of columns in train:28 , test:28
[0]	validation_0-auc:0.85300
[300]	validation_0-auc:0.90068
[600]	validation_0-auc:0.90244
[900]	validation_0-auc:0.90271
[12

In [15]:
train[TARGET].value_counts()

loan_paid_back
1.0    474494
0.0    119500
Name: count, dtype: int64

In [16]:
samp = pd.read_csv(f'{INPUT_DIR}/sample_submission.csv')
samp[TARGET] = test_preds['xgb']
samp.to_csv('baseline_xgb_te_sum_only.csv', index=False)